# 1. Introduction and research question

We study whether oil price increases have different effects on financial markets depending on whether they are more closely associated with demand conditions or with a residual non-demand-related component.

The notebook keeps the empirical strategy deliberately simple. The main analysis is monthly, the oil decomposition is reduced-form, and the predictive regressions are interpreted as forecasting relationships rather than full causal identification.

# 2. Imports and setup

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox

SRC_PATH = Path("../src")
if not SRC_PATH.exists():
    SRC_PATH = Path("src")
if str(SRC_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_PATH.resolve()))

from project_main import (
    DAILY_COLUMNS, MONTHLY_COLUMNS,
    load_data_sheet, aggregate_daily_to_monthly, build_project_variables,
    run_adf_table, run_ljungbox_jb_table, run_diagnostic_residuals,
    ols_from_scratch, ols_summary_df,
    estimate_ma1, ma1_comparison_table,
    decompose_oil_returns, decompose_oil_returns_scratch, add_regime_variables,
    fit_predictive_regression, regression_results_table, interpret_two_component_model,
    rolling_predictive_coefficients,
    granger_pvalue_table, geweke_causality,
    rolling_forecast_comparison, split_sample,
)

warnings.filterwarnings("ignore")
plt.style.use("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_PATH = Path("../data/raw/data_hec_projet_1.xlsx")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/data_hec_projet_1.xlsx")
print("Excel file found:", DATA_PATH.exists())

# 3. Data loading

In [ ]:
daily_raw = load_data_sheet(DATA_PATH, "Daily", DAILY_COLUMNS)
monthly_raw = load_data_sheet(DATA_PATH, "Monthly", MONTHLY_COLUMNS)

print("Daily shape:", daily_raw.shape, "| Range:", daily_raw["date"].min().date(), "to", daily_raw["date"].max().date())
print("Monthly shape:", monthly_raw.shape, "| Range:", monthly_raw["date"].min().date(), "to", monthly_raw["date"].max().date())

# 4. Data cleaning and monthly aggregation

In [ ]:
monthly_market_data = aggregate_daily_to_monthly(daily_raw)
monthly_merged = (
    monthly_market_data.merge(monthly_raw, on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)
print("Merged monthly shape:", monthly_merged.shape)

# 5. Variable construction

In [ ]:
project_df = build_project_variables(monthly_merged)

key_columns = [
    "date", "wti_return", "sp500_return", "hy_change",
    "term_spread", "gold_return", "sp500_realized_vol", "cfnai", "ism_mfg",
]
print("Project dataset shape:", project_df.shape)
display(project_df[key_columns].head(10))

# 6. Descriptive statistics and diagnostics

We combine several diagnostic tools:
- Summary statistics (moments 1–4),
- ADF tests for stationarity,
- Ljung-Box tests for autocorrelation,
- Jarque-Bera tests for normality,
- ACF of squared returns for volatility clustering.

In [ ]:
descriptive_columns = [
    "wti_return", "sp500_return", "hy_change", "term_spread",
    "gold_return", "sp500_realized_vol", "cfnai", "ism_mfg",
]

summary_table = project_df[descriptive_columns].describe().T
summary_table["skew"] = project_df[descriptive_columns].skew()
summary_table["kurtosis"] = project_df[descriptive_columns].kurtosis()
summary_table = summary_table[["count", "mean", "std", "min", "25%", "50%", "75%", "max", "skew", "kurtosis"]].round(4)

print("Table 1. Summary statistics")
display(summary_table)

print("\nTable 2. Correlation matrix")
display(project_df[descriptive_columns].corr().round(3))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, (col, title) in zip(axes, [
    ("wti_return", "Figure 1. Monthly WTI return"),
    ("sp500_return", "Figure 2. Monthly S&P 500 return"),
    ("hy_change", "Figure 3. Monthly HY spread change"),
]):
    ax.plot(project_df["date"], project_df[col])
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
stationarity_columns = ["wti_return", "sp500_return", "hy_change", "term_spread", "gold_return", "cfnai", "ism_mfg"]

print("Table 3a. ADF stationarity tests")
display(run_adf_table(project_df, stationarity_columns))

print("\nTable 3b. Ljung-Box autocorrelation and Jarque-Bera normality tests")
display(run_ljungbox_jb_table(project_df, stationarity_columns))

### Volatility clustering (ACF of squared returns)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["wti_return", "sp500_return", "gold_return"]):
    s = project_df[col].dropna().values
    acf_vals = acf(s**2, nlags=12, fft=True)
    T = len(s)
    ax.bar(range(1, 13), acf_vals[1:13], color="darkorange", width=0.6, alpha=0.8)
    ax.axhline(y=1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8)
    ax.axhline(y=-1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8)
    ax.axhline(y=0, color="black", linewidth=0.5)
    ax.set_title(f"ACF of {col}²")
    ax.set_xlabel("Lag")
plt.suptitle("Figure 4. Volatility clustering — ACF of squared returns", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

for col in ["wti_return", "sp500_return", "gold_return"]:
    lb = acorr_ljungbox(project_df[col].dropna()**2, lags=[6, 12], return_df=True)
    print(f"{col}² — LB(6) p={lb.loc[6,'lb_pvalue']:.4f}, LB(12) p={lb.loc[12,'lb_pvalue']:.4f}")

# 7. Oil shock decomposition

We decompose monthly WTI returns using two approaches:
- **Approach A**: OLS regression on CFNAI. Fitted = demand component, residual = supply component.
- **Approach B**: ISM regime dummy (expansion if ISM > 50, contraction otherwise).

The OLS decomposition is implemented from scratch to show the mechanics.

In [ ]:
# Approach A: OLS from scratch decomposition
decomposition_df, scratch_res = decompose_oil_returns_scratch(
    project_df, oil_return_col="wti_return", activity_col="cfnai", prefix="baseline"
)

print("Table 4. Oil decomposition — OLS from scratch: r_wti = alpha + beta * CFNAI + epsilon")
display(ols_summary_df(scratch_res))
print(f"R² = {scratch_res['R2']:.4f}, T = {scratch_res['T']}")

print("\nResidual diagnostics:")
display(run_diagnostic_residuals(scratch_res["resid"]))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
acf_vals = acf(scratch_res["resid"], nlags=12, fft=True)
T = len(scratch_res["resid"])
ax.bar(range(1, 13), acf_vals[1:13], color="steelblue", width=0.6, alpha=0.8)
ax.axhline(y=1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8)
ax.axhline(y=-1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_title("Figure 5. ACF of decomposition residuals (OLS)")
ax.set_xlabel("Lag")
ax.set_ylabel("ACF")
plt.tight_layout()
plt.show()

## 7b. MA(1) correction

The residuals show significant autocorrelation at lag 6. We estimate the same decomposition with MA(1) errors using maximum likelihood with recursive innovations and OPG standard errors.

In [ ]:
sample = project_df[["wti_return", "cfnai"]].dropna()
ma1_res = estimate_ma1(sample["wti_return"].values, sample["cfnai"].values.reshape(-1, 1), ["cfnai"])

print("Table 5. OLS vs MA(1) MLE comparison")
display(ma1_comparison_table(ma1_res))

print(f"\nMA(1) theta = {ma1_res['theta']:.4f}, OPG SE = {ma1_res['se'][-1]:.4f}")
print(f"Sigma OLS = {np.sqrt(scratch_res['sigma2']):.4f}, Sigma MLE = {ma1_res['sigma']:.4f}")
print("Beta CFNAI moves from {:.4f} (OLS) to {:.4f} (MLE) — decomposition is robust.".format(
    scratch_res["beta"][1], ma1_res["beta"][1]))

print("\nResidual diagnostics after MA(1) correction:")
display(run_diagnostic_residuals(ma1_res["innovations"]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
nlags = 12
acf_ols = acf(scratch_res["resid"], nlags=nlags, fft=True)
acf_ma1 = acf(ma1_res["innovations"], nlags=nlags, fft=True)
lags = np.arange(1, nlags + 1)
T = len(scratch_res["resid"])

ax.stem(lags - 0.15, acf_ols[1:], linefmt="C0-", markerfmt="C0o", basefmt=" ", label="OLS residuals")
ax.stem(lags + 0.15, acf_ma1[1:], linefmt="C1-", markerfmt="C1s", basefmt=" ", label="MA(1) innovations")
ax.axhline(y=1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axhline(y=-1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_title("Figure 6. Residual ACF: OLS vs MA(1) — decomposition of WTI returns")
ax.set_xlabel("Lag")
ax.set_ylabel("Autocorrelation")
ax.legend()
ax.set_xticks(lags)
plt.tight_layout()
plt.show()

In [ ]:
# Approach B: ISM regime variables
decomposition_df = add_regime_variables(decomposition_df, oil_col="wti_return", ism_col="ism_mfg")

n_exp = decomposition_df["expansion"].sum()
n_con = decomposition_df["contraction"].sum()
print(f"Expansion months (ISM > 50): {n_exp} ({100*n_exp/(n_exp+n_con):.1f}%)")
print(f"Contraction months (ISM <= 50): {n_con} ({100*n_con/(n_exp+n_con):.1f}%)")

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(decomposition_df["date"], decomposition_df["wti_return"], label="WTI return", alpha=0.8)
axes[0].plot(decomposition_df["date"], decomposition_df["baseline_oil_demand_component"], label="Demand component", linewidth=1.5)
axes[0].set_title("Figure 7. WTI return and demand-related component")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(decomposition_df["date"], decomposition_df["baseline_oil_supply_component"], label="Supply/residual component", color="firebrick")
axes[1].set_title("Figure 8. Residual oil component (supply/non-demand)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 8. Predictive regressions

We estimate one-month-ahead regressions for S&P 500 returns and HY spread changes. We use both the decomposition model and the regime model, with residual diagnostics after each.

In [ ]:
target_map = {
    "sp500_return": "S&P 500 next-month return",
    "hy_change": "HY next-month change",
}

decomposition_models = {}

for target, label in target_map.items():
    decomp_model, decomp_sample = fit_predictive_regression(
        decomposition_df,
        dependent_col=target,
        predictor_cols=[
            "baseline_oil_demand_component", "baseline_oil_supply_component",
            target, "term_spread", "gold_return", "sp500_realized_vol",
        ],
        horizon=1, cov_type="HC1",
    )
    decomposition_models[target] = decomp_model

    print(f"Decomposition model: {label} (N = {len(decomp_sample)})")
    display(regression_results_table(decomp_model))
    for line in interpret_two_component_model(
        decomp_model, "baseline_oil_demand_component", "baseline_oil_supply_component"):
        print(f"  {line}")
    print("Residual diagnostics:")
    display(run_diagnostic_residuals(decomp_model.resid))
    print()

In [ ]:
for target, label in target_map.items():
    regime_model, regime_sample = fit_predictive_regression(
        decomposition_df,
        dependent_col=target,
        predictor_cols=[
            "oil_expansion", "oil_contraction",
            target, "term_spread", "gold_return", "sp500_realized_vol",
        ],
        horizon=1, cov_type="HC1",
    )
    print(f"Regime model: {label} (N = {len(regime_sample)})")
    display(regression_results_table(regime_model))
    print("Residual diagnostics:")
    display(run_diagnostic_residuals(regime_model.resid))
    print()

## 8b. Stability of predictive coefficients

We estimate the decomposition predictive regression on a rolling 60-month window to check whether the supply/demand asymmetry is stable over time.

In [ ]:
rolling_df = rolling_predictive_coefficients(
    decomposition_df,
    dependent_col="sp500_return",
    predictor_cols=[
        "baseline_oil_demand_component", "baseline_oil_supply_component",
        "sp500_return", "term_spread", "gold_return", "sp500_realized_vol",
    ],
    window=60, horizon=1,
)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for ax, comp, color, title in [
    (axes[0], "baseline_oil_demand_component", "steelblue", "Demand component"),
    (axes[1], "baseline_oil_supply_component", "firebrick", "Supply component"),
]:
    beta_col = f"beta_{comp}"
    se_col = f"se_{comp}"
    ax.plot(rolling_df["date"], rolling_df[beta_col], color=color, linewidth=1.2)
    ax.fill_between(
        rolling_df["date"],
        rolling_df[beta_col] - 2 * rolling_df[se_col],
        rolling_df[beta_col] + 2 * rolling_df[se_col],
        alpha=0.15, color=color,
    )
    ax.axhline(y=0, color="black", linewidth=0.5, linestyle="--")
    ax.set_title(f"Figure 9. Rolling beta — {title} (60-month window)")
    ax.set_ylabel("Coefficient")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 9. Causality tests

We use two approaches:
- **Granger F-tests** (kept from the original notebook),
- **Geweke (1982) causality measures** — directional, instantaneous, and total — with chi-squared tests, as implemented in Class 6.

In [ ]:
# Granger causality (kept from original)
granger_specs = [
    ("baseline_oil_demand_component", "sp500_return"),
    ("baseline_oil_supply_component", "sp500_return"),
    ("baseline_oil_demand_component", "hy_change"),
    ("baseline_oil_supply_component", "hy_change"),
]
granger_tables = [granger_pvalue_table(decomposition_df, c, e, max_lag=3) for c, e in granger_specs]
print("Table 6. Granger causality p-values")
display(pd.concat(granger_tables, ignore_index=True))

In [ ]:
# Geweke causality measures on rolling variances (Class 6 methodology)
# First compute 60-day rolling variances from daily data
daily_returns = daily_raw[["date", "sp500", "wti"]].copy()
daily_returns["r_sp500"] = np.log(daily_returns["sp500"]).diff()
daily_returns["r_wti"] = np.log(daily_returns["wti"]).diff()
daily_returns = daily_returns.dropna()

var_sp = daily_returns["r_sp500"].rolling(60).var()
var_wti = daily_returns["r_wti"].rolling(60).var()
var_df = pd.DataFrame({"sp500_var": var_sp, "wti_var": var_wti}).dropna()

print("Table 7. Geweke causality measures — WTI vs S&P 500 rolling variances")
display(geweke_causality(var_df, "wti_var", "sp500_var", lag=2))

In [ ]:
# Geweke by sub-periods (5-year blocks, Class 6 style)
years = var_df.index.map(lambda x: daily_returns.loc[x, "date"].year if x in daily_returns.index else None)
var_df_dated = var_df.copy()
var_df_dated["year"] = daily_returns.loc[var_df.index, "date"].dt.year.values

blocks = []
for start in range(2000, 2025, 5):
    sub = var_df_dated[(var_df_dated["year"] >= start) & (var_df_dated["year"] < start + 5)][["wti_var", "sp500_var"]]
    if len(sub) < 200:
        continue
    g = geweke_causality(sub, "wti_var", "sp500_var", lag=2)
    wti_to_sp = g[g["measure"].str.contains("wti_var -> sp500_var")]
    if len(wti_to_sp) > 0:
        blocks.append({"period": f"{start}-{start+4}", "C": wti_to_sp["C"].values[0],
                       "statistic": wti_to_sp["statistic"].values[0], "p_value": wti_to_sp["p_value"].values[0]})

print("Table 8. Geweke WTI -> S&P 500 by sub-period")
display(pd.DataFrame(blocks))

# 10. VAR and impulse responses

We estimate a reduced-form VAR with the core variables. We follow the Class 5 methodology: lag selection, stability check, residual diagnostics, orthogonalized IRFs with Monte Carlo confidence bands, and variance decomposition.

In [ ]:
var_columns = ["wti_return", "sp500_return", "hy_change", "term_spread", "gold_return", "ism_mfg"]
var_df = decomposition_df[["date"] + var_columns].dropna().set_index("date")

var_model = VAR(var_df)
lag_selection = var_model.select_order(maxlags=6)
lag_choice = lag_selection.selected_orders["bic"]
if lag_choice == 0:
    lag_choice = max(1, lag_selection.selected_orders["aic"])

print("Table 9. VAR lag selection")
display(pd.DataFrame([lag_selection.selected_orders], index=["selected_lag"]))
print("Chosen lag:", lag_choice)

var_results = var_model.fit(lag_choice)

# Stability check
print(f"\nVAR is stable: {var_results.is_stable()}")

# Residual diagnostics (Ljung-Box on each equation)
print("\nTable 10. Ljung-Box on VAR residuals (lag 12)")
for col in var_columns:
    lb = acorr_ljungbox(var_results.resid[col], lags=[12], return_df=True)
    print(f"  {col}: LB(12) stat={lb.loc[12,'lb_stat']:.2f}, p={lb.loc[12,'lb_pvalue']:.4f}")

In [ ]:
irf = var_results.irf(24)
irf_err = irf.errband_mc(orth=True, repl=300, signif=0.05, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, response, title in [
    (axes[0], "sp500_return", "Figure 10. IRF: S&P 500 response to oil shock"),
    (axes[1], "hy_change", "Figure 11. IRF: HY spread response to oil shock"),
]:
    impulse_idx = var_columns.index("wti_return")
    response_idx = var_columns.index(response)
    irf_vals = irf.orth_irfs[:, response_idx, impulse_idx]
    lower = irf_err[0][:, response_idx, impulse_idx]
    upper = irf_err[1][:, response_idx, impulse_idx]

    ax.plot(range(25), irf_vals, color="navy", linewidth=1.5)
    ax.fill_between(range(25), lower, upper, alpha=0.15, color="steelblue")
    ax.axhline(y=0, color="black", linewidth=0.5, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Months")
    ax.set_ylabel("Response")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fevd = var_results.fevd(12)

print("Table 11. Forecast error variance decomposition at horizon 12")
fevd_table = pd.DataFrame(fevd.decomp[11], index=var_columns, columns=var_columns).round(4)
display(fevd_table[["wti_return"]].rename(columns={"wti_return": "share_explained_by_oil"}))

# 11. Out-of-sample forecasting

We compare three approaches:
- Historical mean benchmark,
- Raw oil model (single WTI return),
- Decomposition model (demand + supply components).

We also add a VAR forecast exercise following the Class 5 methodology.

In [ ]:
# OLS rolling forecast comparison (kept from original)
for target, label in target_map.items():
    _, metrics_df = rolling_forecast_comparison(
        decomposition_df,
        target_col=target, raw_oil_col="wti_return",
        demand_col="baseline_oil_demand_component",
        supply_col="baseline_oil_supply_component",
        controls=["term_spread", "gold_return", "sp500_realized_vol"],
        start_share=0.6,
    )
    print(f"Table 12. Forecast comparison — {label}")
    display(metrics_df)
    print()

In [ ]:
# VAR out-of-sample forecast (Class 5 style)
split_date = "2020-01-01"
train_var = var_df[var_df.index < split_date]
test_var = var_df[var_df.index >= split_date]

if len(test_var) > 0 and len(train_var) > lag_choice:
    var_train_model = VAR(train_var).fit(lag_choice)
    n_forecast = min(len(test_var), 12)
    forecast_vals = var_train_model.forecast(train_var.values[-lag_choice:], steps=n_forecast)
    forecast_df = pd.DataFrame(forecast_vals, index=test_var.index[:n_forecast], columns=var_columns)

    print("Table 13. VAR forecast RMSE (train < 2020, test >= 2020)")
    for col in ["sp500_return", "hy_change"]:
        actual = test_var[col].iloc[:n_forecast]
        pred = forecast_df[col]
        rmse = np.sqrt(np.mean((actual - pred)**2))
        print(f"  {col}: RMSE = {rmse:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, col, title in [
        (axes[0], "sp500_return", "Figure 12. VAR forecast vs realized — S&P 500"),
        (axes[1], "hy_change", "Figure 13. VAR forecast vs realized — HY change"),
    ]:
        ax.plot(test_var.index[:n_forecast], test_var[col].iloc[:n_forecast], label="Realized", marker="o", markersize=4)
        ax.plot(forecast_df.index, forecast_df[col], label="VAR forecast", marker="s", markersize=4)
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 12. Robustness checks

Three robustness tests:
1. Brent instead of WTI,
2. MSCI EM instead of S&P 500,
3. Sub-period analysis (pre-2008, 2008–2014, post-2015).

In [ ]:
# Brent robustness
brent_df, _ = decompose_oil_returns(decomposition_df, oil_return_col="brent_return", activity_col="cfnai", prefix="brent")
brent_model, brent_sample = fit_predictive_regression(
    brent_df, dependent_col="sp500_return",
    predictor_cols=["brent_oil_demand_component", "brent_oil_supply_component",
                    "sp500_return", "term_spread", "gold_return", "sp500_realized_vol"],
    horizon=1, cov_type="HC1",
)

# MSCI EM robustness
em_model, em_sample = fit_predictive_regression(
    decomposition_df, dependent_col="msci_em_return",
    predictor_cols=["baseline_oil_demand_component", "baseline_oil_supply_component",
                    "msci_em_return", "term_spread", "gold_return", "sp500_realized_vol"],
    horizon=1, cov_type="HC1",
)

robustness_rows = []
for name, model, demand_name, supply_name in [
    ("Brent instead of WTI", brent_model, "brent_oil_demand_component", "brent_oil_supply_component"),
    ("MSCI EM instead of S&P", em_model, "baseline_oil_demand_component", "baseline_oil_supply_component"),
]:
    robustness_rows.append({
        "check": name,
        "demand_coef": model.params.get(demand_name, np.nan),
        "demand_pval": model.pvalues.get(demand_name, np.nan),
        "supply_coef": model.params.get(supply_name, np.nan),
        "supply_pval": model.pvalues.get(supply_name, np.nan),
    })

print("Table 14. Robustness — alternative oil benchmark and equity market")
display(pd.DataFrame(robustness_rows).round(4))

In [ ]:
# Sub-period analysis
sub_periods = [
    ("1990-2007", "1990-01-01", "2008-01-01"),
    ("2008-2014", "2008-01-01", "2015-01-01"),
    ("2015-2026", "2015-01-01", "2027-01-01"),
]

sub_rows = []
for label, start, end in sub_periods:
    sub = decomposition_df[(decomposition_df["date"] >= start) & (decomposition_df["date"] < end)]
    if len(sub) < 30:
        continue
    sub_model, sub_sample = fit_predictive_regression(
        sub, dependent_col="sp500_return",
        predictor_cols=["baseline_oil_demand_component", "baseline_oil_supply_component",
                        "sp500_return", "term_spread", "gold_return", "sp500_realized_vol"],
        horizon=1, cov_type="HC1",
    )
    sub_rows.append({
        "period": label, "N": len(sub_sample),
        "demand_coef": sub_model.params.get("baseline_oil_demand_component", np.nan),
        "demand_pval": sub_model.pvalues.get("baseline_oil_demand_component", np.nan),
        "supply_coef": sub_model.params.get("baseline_oil_supply_component", np.nan),
        "supply_pval": sub_model.pvalues.get("baseline_oil_supply_component", np.nan),
    })

print("Table 15. Sub-period analysis — predictive regression coefficients")
display(pd.DataFrame(sub_rows).round(4))

# 13. Main conclusions

The main message of the notebook should be read in a reduced-form sense. The fitted oil component is interpreted as demand-related, while the residual component is interpreted as non-demand-related. The project provides a transparent and coherent empirical framework rather than a full structural identification strategy.

In [ ]:
print("Main takeaway for S&P 500 returns")
for line in interpret_two_component_model(
    decomposition_models["sp500_return"],
    "baseline_oil_demand_component", "baseline_oil_supply_component"):
    print(f"  {line}")

print("\nMain takeaway for HY changes")
for line in interpret_two_component_model(
    decomposition_models["hy_change"],
    "baseline_oil_demand_component", "baseline_oil_supply_component"):
    print(f"  {line}")